# SignSpeak — Dedicated ASL Recognizer (WLASL-100 / 300)

**Purpose:** Train a small isolated-sign classifier on MediaPipe Holistic keypoints to run **alongside** Gemini on [SignSpeak](https://signspeak-asl-poc.vercel.app/demo).

**License / ethics (read first):**
- WLASL is licensed under **Microsoft C-UDA 1.0** — computational / research use only. **Not for commercial use of the dataset.**
- You must agree to C-UDA before downloading samples: https://github.com/dxli94/WLASL
- This notebook is for a DECA EIP **research / non-commercial** POC.
- Trained model weights (Results under C-UDA) are generally freer to share than the raw videos — still credit WLASL.

**Runtime:** Colab free GPU (T4). Use **Runtime → Change runtime type → GPU**.

**Pipeline:** WLASL metadata → subset 100/300 classes → download a few videos per class → MediaPipe Holistic keypoints → TCN classifier → export `label_map.json` + `wlasl100_tcn.pt`.


In [ ]:
# GPU check
!nvidia-smi -L || true
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


In [ ]:
# Install deps (Colab)
%pip -q install mediapipe opencv-python-headless tqdm scikit-learn
# torch is preinstalled on Colab


In [ ]:
# Accept C-UDA reminder
print('''Before continuing, open https://github.com/dxli94/WLASL and read C-UDA-1.0.
Type YES if this is for research / non-commercial computational use only.''')
AGREE = input('Agree to C-UDA research use? Type YES: ').strip()
assert AGREE.upper() == 'YES', 'Stop: C-UDA agreement required.'


In [ ]:
from pathlib import Path
import json, os, random, math, urllib.request, subprocess, sys
from collections import defaultdict

ROOT = Path('/content/signspeak_asl')
DATA = ROOT / 'data'
META = DATA / 'meta'
VIDEOS = DATA / 'videos'
KP = DATA / 'keypoints'
ART = ROOT / 'artifacts'
for p in [META, VIDEOS, KP, ART]:
    p.mkdir(parents=True, exist_ok=True)

WLASL_JSON_URL = 'https://raw.githubusercontent.com/dxli94/WLASL/master/start_kit/WLASL_v0.3.json'
# Fallback mirrors if path differs
CANDIDATES = [
    WLASL_JSON_URL,
    'https://raw.githubusercontent.com/dxli94/WLASL/master/WLASL_v0.3.json',
]

meta_path = META / 'WLASL_v0.3.json'
if not meta_path.exists():
    last_err = None
    for url in CANDIDATES:
        try:
            print('Downloading', url)
            urllib.request.urlretrieve(url, meta_path)
            break
        except Exception as e:
            last_err = e
            print('failed', e)
    else:
        raise RuntimeError(f'Could not download WLASL JSON: {last_err}')

raw = json.loads(meta_path.read_text())
print('entries', len(raw), 'type', type(raw))
print('sample keys', list(raw[0].keys()) if isinstance(raw, list) else list(raw.keys())[:10])


In [ ]:
# Build WLASL-100 or WLASL-300 subset (most instances first)
SUBSET = int(os.environ.get('WLASL_SUBSET', '100'))  # set to 300 if you want
assert SUBSET in (100, 300, 1000, 2000)

# WLASL JSON is a list of gloss entries with instances
# Each: {"gloss": str, "instances": [{"video_id", "url", "split", ...}, ...]}

def instance_count(entry):
    return len(entry.get('instances') or [])

ranked = sorted(raw, key=instance_count, reverse=True)
subset = ranked[:SUBSET]
glosses = [e['gloss'] for e in subset]
label_to_idx = {g: i for i, g in enumerate(glosses)}
idx_to_label = {i: g for g, i in label_to_idx.items()}
(ART / 'label_map.json').write_text(json.dumps({'label_to_idx': label_to_idx, 'idx_to_label': {str(k): v for k,v in idx_to_label.items()}, 'subset': SUBSET}, indent=2))
print('subset', SUBSET, 'top glosses', glosses[:10])


In [ ]:
# Download a capped number of videos per gloss (keeps Colab free-tier feasible)
# Prefer instances that already have a direct URL field.
MAX_PER_CLASS = int(os.environ.get('MAX_PER_CLASS', '8'))
MAX_TOTAL = int(os.environ.get('MAX_TOTAL', '600'))

manifest = []
total = 0
for entry in subset:
    gloss = entry['gloss']
    insts = entry.get('instances') or []
    random.shuffle(insts)
    kept = 0
    for inst in insts:
        if kept >= MAX_PER_CLASS or total >= MAX_TOTAL:
            break
        url = inst.get('url') or inst.get('video_url')
        vid = str(inst.get('video_id') or inst.get('id') or f'{gloss}_{kept}')
        split = inst.get('split') or 'train'
        if not url:
            continue
        out = VIDEOS / f'{vid}.mp4'
        if not out.exists():
            try:
                urllib.request.urlretrieve(url, out)
            except Exception as e:
                print('skip', vid, e)
                continue
        if out.exists() and out.stat().st_size > 10_000:
            manifest.append({'video_id': vid, 'path': str(out), 'gloss': gloss, 'label': label_to_idx[gloss], 'split': split})
            kept += 1
            total += 1

(META / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print('downloaded clips', len(manifest), 'classes covered', len({m['gloss'] for m in manifest}))
if len(manifest) < 20:
    print('''WARNING: Few videos downloaded. WLASL often requires their video_downloader + YouTube.
If this happens, clone https://github.com/dxli94/WLASL and run their start_kit/video_downloader.py,
then copy mp4s into''', VIDEOS)


In [ ]:
# MediaPipe Holistic landmark extraction → fixed-length tensors
import cv2
import numpy as np
from tqdm import tqdm

# MediaPipe Tasks Holistic is version-sensitive; use classic holistic if available.
try:
    import mediapipe as mp
    holistic = mp.solutions.holistic.Holistic(static_image_mode=False, model_complexity=0, refine_face_landmarks=False)
    USE_CLASSIC = True
except Exception as e:
    print('classic holistic unavailable', e)
    USE_CLASSIC = False
    holistic = None

NUM_FRAMES = 64
# pose 33 + left hand 21 + right hand 21 = 75 joints (xyz)
NUM_JOINTS = 75

def landmarks_from_results(results):
    def pack(lms, n):
        if lms is None:
            return np.zeros((n, 3), dtype=np.float32)
        arr = np.array([[p.x, p.y, p.z] for p in lms.landmark], dtype=np.float32)
        if arr.shape[0] != n:
            out = np.zeros((n, 3), dtype=np.float32)
            out[:min(n, arr.shape[0])] = arr[:n]
            return out
        return arr
    pose = pack(results.pose_landmarks, 33)
    lh = pack(results.left_hand_landmarks, 21)
    rh = pack(results.right_hand_landmarks, 21)
    return np.concatenate([pose, lh, rh], axis=0)  # (75,3)

def extract_video(path, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        if USE_CLASSIC:
            res = holistic.process(rgb)
            frames.append(landmarks_from_results(res))
        else:
            # fallback: zeros (should not happen on Colab with mediapipe)
            frames.append(np.zeros((NUM_JOINTS, 3), dtype=np.float32))
    cap.release()
    if not frames:
        return None
    arr = np.stack(frames, axis=0)  # (T,J,3)
    T = arr.shape[0]
    if T == num_frames:
        out = arr
    elif T > num_frames:
        idx = np.linspace(0, T - 1, num_frames).astype(int)
        out = arr[idx]
    else:
        pad = np.repeat(arr[-1:], num_frames - T, axis=0)
        out = np.concatenate([arr, pad], axis=0)
    # normalize: center on pose mid-hip if present
    mid = out[:, 23:25, :].mean(axis=1, keepdims=True)  # approx
    out = out - mid
    return out.astype(np.float32)

kp_manifest = []
for row in tqdm(manifest):
    dest = KP / f"{row['video_id']}.npy"
    if not dest.exists():
        arr = extract_video(row['path'])
        if arr is None:
            continue
        np.save(dest, arr)
    kp_manifest.append({**row, 'keypoints': str(dest)})

(META / 'kp_manifest.json').write_text(json.dumps(kp_manifest, indent=2))
print('keypoints', len(kp_manifest))


In [ ]:
# Dataset + lightweight TCN classifier (Colab-friendly alternative to full ST-GCN)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class LandmarkDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        x = np.load(r['keypoints'])  # (T,J,3)
        x = torch.from_numpy(x).permute(2, 0, 1).contiguous()  # (C,T,J)
        y = torch.tensor(r['label'], dtype=torch.long)
        return x, y

rows = json.loads((META / 'kp_manifest.json').read_text())
# Prefer official splits when present
train_rows = [r for r in rows if r.get('split') == 'train']
val_rows = [r for r in rows if r.get('split') in ('val', 'test')]
if len(train_rows) < 5 or len(val_rows) < 2:
    labels = [r['label'] for r in rows]
    train_rows, val_rows = train_test_split(rows, test_size=0.2, random_state=42, stratify=labels if len(set(labels))>1 else None)

print('train', len(train_rows), 'val', len(val_rows), 'classes', SUBSET)

class TemporalConvNet(nn.Module):
    '''Small TCN over flattened joints — good free-tier baseline.
    Swap for ST-GCN later using the same (C,T,J) tensors.'''
    def __init__(self, num_classes, channels=3, joints=NUM_JOINTS, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(channels * joints, hidden, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden, hidden, kernel_size=3, padding=2, dilation=2),
            nn.ReLU(),
            nn.Conv1d(hidden, hidden, kernel_size=3, padding=4, dilation=4),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.fc = nn.Linear(hidden, num_classes)
    def forward(self, x):  # (B,C,T,J)
        b, c, t, j = x.shape
        x = x.view(b, c * j, t)
        x = self.net(x).squeeze(-1)
        return self.fc(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TemporalConvNet(num_classes=SUBSET).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
crit = nn.CrossEntropyLoss()
train_loader = DataLoader(LandmarkDataset(train_rows), batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(LandmarkDataset(val_rows), batch_size=16, shuffle=False)

def run_epoch(loader, train=True):
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = crit(logits, y)
        if train:
            opt.zero_grad(); loss.backward(); opt.step()
        loss_sum += loss.item() * y.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return loss_sum / max(total,1), correct / max(total,1)

EPOCHS = int(os.environ.get('EPOCHS', '20'))
best = 0.0
ckpt = ART / f'wlasl{SUBSET}_tcn.pt'
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, True)
    va_loss, va_acc = run_epoch(val_loader, False)
    print(f'epoch {epoch:02d}  train {tr_acc:.3f}  val {va_acc:.3f}')
    if va_acc >= best:
        best = va_acc
        torch.save({'model': model.state_dict(), 'subset': SUBSET, 'num_joints': NUM_JOINTS, 'num_frames': NUM_FRAMES, 'label_to_idx': label_to_idx}, ckpt)
print('best_val', best, 'saved', ckpt)


In [ ]:
# Inference helper (single clip path)
@torch.no_grad()
def predict_video(path, threshold=0.55):
    arr = extract_video(path)
    if arr is None:
        return {'ok': False, 'reason': 'no_frames'}
    x = torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).to(device)
    logits = model(x)
    prob = torch.softmax(logits, dim=-1)[0]
    conf, idx = prob.max(0)
    gloss = idx_to_label[int(idx)]
    return {
        'ok': True,
        'gloss': gloss,
        'confidence': float(conf),
        'below_threshold': float(conf) < threshold,
        'fallback': 'gemini' if float(conf) < threshold else 'dedicated',
    }

# quick smoke on one training clip if available
if train_rows:
    print(predict_video(train_rows[0]['path']))
print('Download artifacts from', ART)


## Next: wire into SignSpeak

1. Download from Colab files sidebar: `artifacts/wlasl100_tcn.pt`, `artifacts/label_map.json`.
2. Copy `schemas/asl-key-vocabulary.example.json` from the repo and fill `wlaslClassId` / glosses to match `label_map.json`.
3. Follow `docs/asl-dedicated-routing.md` for `/api/interpret` hybrid routing (dedicated first → Gemini fallback).
4. Keep Gemini for open vocabulary (names, lyrics, OOV). Dedicated model owns the curated key list only.

### Optional ST-GCN upgrade
Replace `TemporalConvNet` with an ST-GCN that consumes the same `(B,C,T,J)` tensors and MediaPipe joint adjacency (pose+hands). Training loop and exports stay identical.

